# Unidad 5: Web Scraping y Automatización de Procesos Digitales
**Materia:** Programación — Licenciatura en Negocios Digitales (3º año)
**Institución:** Universidad del Museo Social Argentino (UMSA)

---

## Introducción y Contexto de Negocio

En los negocios digitales, las decisiones se toman en base a **datos**. Sin embargo, mucha de la información valiosa no está expuesta en una API ordenada: está distribuida por la web pública. Por ejemplo, los precios de tus competidores, las opiniones de los usuarios en foros de turismo, las propiedades en portales inmobiliarios, o las ofertas de empleo en LinkedIn.

El **Web Scraping** es la técnica para extraer automáticamente información estructurada a partir del código HTML de páginas web.

En esta unidad aprenderemos a:
1. **Analizar la estructura del DOM** de una página web (HTML, CSS Selectors y XPath).
2. **Extraer datos de páginas estáticas** de forma rápida utilizando **BeautifulSoup** y `requests`.
3. **Automatizar la navegación en sitios dinámicos** cargados con JavaScript mediante **Playwright**.
4. **Respetar las buenas prácticas éticas y legales**: el control de frecuencia (rate limiting), el uso de User-Agents y la lectura de archivos `robots.txt`.


## 1. La Estructura del DOM: El Mapa del Sitio Web

Una página web se compone de HTML (estructura), CSS (diseño) y JavaScript (dinamismo). El navegador organiza el HTML en un árbol jerárquico de objetos llamado **DOM (Document Object Model)**.

Para extraer datos, debemos indicarle a nuestros scripts cómo encontrar los elementos en el árbol usando:
- **CSS Selectors**: Sintaxis compacta similar a las reglas de estilo (ej. `div.producto p.precio`).
- **XPath**: Un lenguaje de rutas para navegar a través de los nodos del árbol (ej. `//div[@class="producto"]/p`).


In [1]:
import pandas as pd
from bs4 import BeautifulSoup
from lxml import etree

# 1. HTML simulado del competidor
html_tienda = """
<!DOCTYPE html>
<html>
<head><title>Marketplace de Tecnología</title></head>
<body>
    <div id="catalogo">
        <div class="card-producto" data-id="101">
            <h3 class="nombre-prod">Laptop UltraBook 14"</h3>
            <span class="precio-actual">$1250.00</span>
            <span class="categoria">Computadoras</span>
            <p class="stock disponible">En Stock</p>
        </div>
        <div class="card-producto" data-id="102">
            <h3 class="nombre-prod">Monitor Gamer 27" 144Hz</h3>
            <span class="precio-actual">$380.00</span>
            <span class="categoria">Monitores</span>
            <p class="stock agotado">Agotado</p>
        </div>
        <div class="card-producto" data-id="103">
            <h3 class="nombre-prod">Teclado Mecánico RGB</h3>
            <span class="precio-actual">$95.50</span>
            <span class="categoria">Accesorios</span>
            <p class="stock disponible">En Stock</p>
        </div>
    </div>
</body>
</html>
"""

# A) Parseo y extracción mediante CSS Selectors (BeautifulSoup)
soup = BeautifulSoup(html_tienda, "html.parser")
productos_css = []

for card in soup.select("div.card-producto"):
    nombre = card.select_one("h3.nombre-prod").text.strip()
    precio = float(card.select_one("span.precio-actual").text.replace("$", ""))
    disponible = "disponible" in card.select_one("p.stock")["class"]

    productos_css.append({"nombre": nombre, "precio": precio, "disponible": disponible})

print("=== Resultado usando CSS Selectors ===")
print(pd.DataFrame(productos_css))

# B) Parseo y extracción mediante XPath (lxml)
dom = etree.HTML(html_tienda)
nombres_xpath = dom.xpath('//div[@class="card-producto"]/h3[@class="nombre-prod"]/text()')
precios_xpath = dom.xpath('//div[@class="card-producto"]/span[@class="precio-actual"]/text()')

print("\n=== Resultado usando XPath ===")
for nom, pre in zip(nombres_xpath, precios_xpath):
    print(f"Producto: {nom} | Precio: {pre}")

=== Resultado usando CSS Selectors ===
                    nombre  precio  disponible
0     Laptop UltraBook 14"  1250.0        True
1  Monitor Gamer 27" 144Hz   380.0       False
2     Teclado Mecánico RGB    95.5        True

=== Resultado usando XPath ===
Producto: Laptop UltraBook 14" | Precio: $1250.00
Producto: Monitor Gamer 27" 144Hz | Precio: $380.00
Producto: Teclado Mecánico RGB | Precio: $95.50


## 2. Scraping Estático con BeautifulSoup

Si la página web renderiza todo su contenido directamente en el servidor (HTML estático), podemos descargarla con `requests` y procesarla rápidamente con `BeautifulSoup`.


In [2]:
from bs4 import BeautifulSoup
import requests

# Simulamos una página web de e-commerce mediante un string HTML
html_simulado = """
<html>
    <head><title>Tienda de Computación</title></head>
    <body>
        <h1>Catálogo de Ofertas</h1>
        <div class="listado-productos">
            <div class="producto" id="p1">
                <h2 class="titulo">Mouse Inalámbrico Logitech</h2>
                <span class="precio">$4500.00</span>
                <p class="estado disponible">En Stock</p>
            </div>
            <div class="producto" id="p2">
                <h2 class="titulo">Teclado Mecánico RGB</h2>
                <span class="precio">$12000.00</span>
                <p class="estado sin-stock">Sin Stock</p>
            </div>
            <div class="producto" id="p3">
                <h2 class="titulo">Monitor Gamer 24"</h2>
                <span class="precio">$85000.00</span>
                <p class="estado disponible">En Stock</p>
            </div>
        </div>
    </body>
</html>
"""

# Cargamos el HTML en BeautifulSoup
soup = BeautifulSoup(html_simulado, "html.parser")

# Buscamos todos los bloques de productos
productos = soup.select("div.producto")

print(f"Productos encontrados: {len(productos)}\n")

for p in productos:
    titulo = p.select_one(".titulo").text
    # Limpiamos el signo de pesos y convertimos el precio a float
    precio_str = p.select_one(".precio").text
    precio = float(precio_str.replace("$", ""))
    estado = p.select_one(".estado").text

    print(f"Producto: {titulo} | Precio: ${precio:.2f} | Estado: {estado}")


Productos encontrados: 3

Producto: Mouse Inalámbrico Logitech | Precio: $4500.00 | Estado: En Stock
Producto: Teclado Mecánico RGB | Precio: $12000.00 | Estado: Sin Stock
Producto: Monitor Gamer 24" | Precio: $85000.00 | Estado: En Stock


Ejercicio 2: Rate Limiting, User-Agents y Ética (robots.txt)
Consigna:
Implementar un bucle de peticiones HTTP utilizando requests que incluya rotación de User-Agents, demoras aleatorias entre peticiones para respetar el rate limiting y una función de verificación previa del archivo robots.txt.

In [3]:
import requests
import time
import random
from urllib.parse import urlparse
from urllib.robotparser import RobotFileParser

# Lista de User-Agents reales para simular distintos navegadores
USER_AGENTS = [
    "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/115.0.0.0 Safari/537.36",
    "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/605.1.15 (KHTML, like Gecko) Version/16.0 Safari/605.1.15",
    "Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/114.0.0.0 Safari/537.36"
]

def es_url_permitida(url: str, user_agent: str = "*") -> bool:
    """Verifica si la URL puede ser rastreada según el robots.txt del sitio."""
    parsed_url = urlparse(url)
    robots_url = f"{parsed_url.scheme}://{parsed_url.netloc}/robots.txt"

    rp = RobotFileParser()
    rp.set_url(robots_url)
    try:
        rp.read()
        return rp.can_fetch(user_agent, url)
    except Exception:
        # Si no se puede leer el robots.txt, se asume precaución
        return True

def scraping_etico_simulado(urls: list):
    for idx, url in enumerate(urls, 1):
        if not es_url_permitida(url):
            print(f"[BLOQUEADO] La URL {url} no está permitida por el robots.txt del sitio.")
            continue

        # Configuración de cabeceras con User-Agent aleatorio
        headers = {"User-Agent": random.choice(USER_AGENTS)}
        print(f"\n[Petición #{idx}] Solicitando: {url}")
        print(f"User-Agent utilizado: {headers['User-Agent'][:50]}...")

        try:
            respuesta = requests.get(url, headers=headers, timeout=5)
            print(f"Estado de Respuesta: {respuesta.status_code}")
        except Exception as e:
            print(f"Error al conectar: {e}")

        # Rate limiting: Espera aleatoria entre 1.5 y 3.0 segundos
        tiempo_espera = random.uniform(1.5, 3.0)
        print(f"Esperando {tiempo_espera:.2f} segundos para no saturar el servidor...")
        time.sleep(tiempo_espera)

# Ejecución de prueba con un sitio seguro para scraping
urls_a_procesar = [
    "https://quotes.toscrape.com/page/1/",
    "https://quotes.toscrape.com/page/2/"
]
scraping_etico_simulado(urls_a_procesar)


[Petición #1] Solicitando: https://quotes.toscrape.com/page/1/
User-Agent utilizado: Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) Ap...
Estado de Respuesta: 200
Esperando 1.91 segundos para no saturar el servidor...

[Petición #2] Solicitando: https://quotes.toscrape.com/page/2/
User-Agent utilizado: Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36...
Estado de Respuesta: 200
Esperando 2.33 segundos para no saturar el servidor...


## 3. Scraping Dinámico con Playwright

Muchas páginas web modernas son **Single Page Applications (SPAs)** creadas con React, Vue o Angular. En estos sitios, el HTML inicial que devuelve el servidor está prácticamente vacío y el contenido se carga dinámicamente mediante JavaScript realizando peticiones en segundo plano.

Si usamos `requests.get()` en estos sitios, no obtendremos los datos de los productos. Necesitamos un **navegador real automatizado** que ejecute el JavaScript. Para esto usamos **Playwright**.

### Instalación de Playwright en Google Colab


In [10]:
!pip install playwright --quiet
!playwright install chromium
!playwright install-deps chromium

Installing dependencies...
Hit:1 https://cli.github.com/packages stable InRelease
Hit:2 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease
Hit:3 https://r2u.stat.illinois.edu/ubuntu jammy InRelease
Hit:4 http://archive.ubuntu.com/ubuntu jammy InRelease
Hit:5 http://security.ubuntu.com/ubuntu jammy-security InRelease
Hit:6 http://archive.ubuntu.com/ubuntu jammy-updates InRelease
Hit:7 http://archive.ubuntu.com/ubuntu jammy-backports InRelease
Hit:8 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Hit:9 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Reading package lists... Done
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
fonts-freefont-ttf is already the newest version (20120503-10build1

In [9]:
import subprocess
import asyncio
from playwright.async_api import async_playwright

# Aseguramos la instalación de los navegadores de Playwright desde Python
try:
    subprocess.run(["playwright", "install", "chromium"], check=True)
    subprocess.run(["playwright", "install-deps", "chromium"], check=True)
except Exception as e:
    print(f"Aviso durante la instalación de navegadores: {e}")

async def ejecutar_scraping_dinamico():
    async with async_playwright() as p:
        # Lanzamos Chromium en modo headless
        browser = await p.chromium.launch(headless=True)
        page = await browser.new_page()

        # Navegamos a la página con contenido dinámico
        await page.goto("https://quotes.toscrape.com/js/")

        # Esperamos a que los elementos carguen en el DOM
        await page.wait_for_selector(".quote")

        citas_elementos = await page.query_selector_all(".quote")
        print(f"Citas cargadas con JS encontradas: {len(citas_elementos)}\n")

        for cita_el in citas_elementos[:3]:
            texto_el = await cita_el.query_selector(".text")
            autor_el = await cita_el.query_selector(".author")

            texto = await texto_el.inner_text() if texto_el else ""
            autor = await autor_el.inner_text() if autor_el else ""

            print(f"Cita: '{texto}' - Autor: {autor}")

        await browser.close()

# Ejecución asíncrona dentro del notebook
await ejecutar_scraping_dinamico()

Citas cargadas con JS encontradas: 10

Cita: '“The world as we have created it is a process of our thinking. It cannot be changed without changing our thinking.”' - Autor: Albert Einstein
Cita: '“It is our choices, Harry, that show what we truly are, far more than our abilities.”' - Autor: J.K. Rowling
Cita: '“There are only two ways to live your life. One is as though nothing is a miracle. The other is as though everything is a miracle.”' - Autor: Albert Einstein


### Ejecución de Playwright de forma Asíncrona (Apto para Jupyter)

Debido a que Google Colab ya corre dentro de un bucle de eventos asíncrono (`asyncio`), en notebooks debemos utilizar la versión asíncrona de Playwright (`async_api`).


In [11]:
import asyncio
from playwright.async_api import async_playwright

async def ejecutar_scraping_dinamico():
    async with async_playwright() as p:
        # Lanzamos el navegador en modo headless
        browser = await p.chromium.launch(headless=True)
        page = await browser.new_page()

        # Navegamos a una página de prueba con carga dinámica
        await page.goto("https://quotes.toscrape.com/js/")

        # Esperamos a que el elemento con las citas se renderice en el DOM
        await page.wait_for_selector(".quote")

        # Extraemos los datos usando selectores CSS
        citas_elementos = await page.query_selector_all(".quote")
        print(f"Citas cargadas con JS encontradas: {len(citas_elementos)}\n")

        for cita_el in citas_elementos[:3]: # Mostramos las primeras 3
            texto = await (await cita_el.query_selector(".text")).inner_text()
            autor = await (await cita_el.query_selector(".author")).inner_text()
            print(f"Cita: '{texto}' - Autor: {autor}")

        await browser.close()

# Ejecutamos la función asíncrona dentro del notebook
await ejecutar_scraping_dinamico()


Citas cargadas con JS encontradas: 10

Cita: '“The world as we have created it is a process of our thinking. It cannot be changed without changing our thinking.”' - Autor: Albert Einstein
Cita: '“It is our choices, Harry, that show what we truly are, far more than our abilities.”' - Autor: J.K. Rowling
Cita: '“There are only two ways to live your life. One is as though nothing is a miracle. The other is as though everything is a miracle.”' - Autor: Albert Einstein


## 4. Aspectos Éticos, Legales y Técnicos

Hacer scraping masivo sin control puede saturar el servidor de una empresa pequeña (equivalente a un ataque de denegación de servicio DDoS) o violar sus términos de uso.

### Buenas Prácticas:
1. **Revisar el `robots.txt`**: Archivo que indica qué carpetas del sitio web se permite scrapear (ej. `https://tureservas.com/robots.txt`).
2. **Rotar User-Agents**: El `User-Agent` es un header HTTP que identifica el navegador. Si mandamos miles de peticiones con el User-Agent por defecto de `requests` (`python-requests/X.X.X`), los firewalls nos bloquearán inmediatamente. Debemos simular ser un navegador humano.
3. **Respetar el Rate Limiting**: Añadir retrasos temporales (`time.sleep()` o esperas asíncronas) entre peticiones para no saturar al servidor.


In [12]:
import time
import random

user_agents = [
    "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/115.0.0.0 Safari/537.36",
    "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/605.1.15 (KHTML, like Gecko) Version/16.0 Safari/605.1.15",
    "Mozilla/5.0 (Linux; Android 10; K) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/114.0.0.0 Mobile Safari/537.36"
]

# Simulación de un bucle de scraping ético
for i in range(3):
    headers = {"User-Agent": random.choice(user_agents)}
    print(f"Petición #{i+1} enviada con User-Agent: {headers['User-Agent'][:60]}...")

    # Realizar petición simulada
    # respuesta = requests.get(url, headers=headers)

    # Espera aleatoria para imitar comportamiento humano y no sobrecargar
    espera = random.uniform(1.0, 3.0)
    print(f"Esperando {espera:.2f} segundos antes de la siguiente petición...")
    time.sleep(espera)


Petición #1 enviada con User-Agent: Mozilla/5.0 (Linux; Android 10; K) AppleWebKit/537.36 (KHTML...
Esperando 2.56 segundos antes de la siguiente petición...
Petición #2 enviada con User-Agent: Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36...
Esperando 2.17 segundos antes de la siguiente petición...
Petición #3 enviada con User-Agent: Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/...
Esperando 2.03 segundos antes de la siguiente petición...


Ejercicio 4 (Integrador): Competitive Intelligence — Scraping y Persistencia en Base de Datos Relacional
Consigna:
Construir un script completo de Inteligencia Competitiva que:

Genere/descargue datos de productos de un competidor.

Limpie y transforme los tipos de datos (precios a flotantes, nombres normalizados).

Persista los resultados en una base de datos SQLite usando Pandas.

Realice una consulta SQL para identificar los productos que requieren ajuste de precio para competir.

In [13]:
import sqlite3
import pandas as pd
from bs4 import BeautifulSoup

# 1. Simulación del HTML recibido del competidor
html_competidor_tienda = """
<div class="catalogo-competidor">
    <div class="item">
        <h2 class="title">  Silla Ergonómica Gamer Pro  </h2>
        <span class="price">$ 45,999.00</span>
        <span class="brand">Marca Alpha</span>
    </div>
    <div class="item">
        <h2 class="title">Escritorio Regulable Motorizado</h2>
        <span class="price">$ 120,500.50</span>
        <span class="brand">Marca Alpha</span>
    </div>
    <div class="item">
        <h2 class="title">Soporte Monitor Dual</h2>
        <span class="price">$ 18,200.00</span>
        <span class="brand">Marca Beta</span>
    </div>
</div>
"""

# 2. Extracción y Limpieza con BeautifulSoup
soup = BeautifulSoup(html_competidor_tienda, "html.parser")
datos_preparados = []

for item in soup.select("div.item"):
    nombre = item.select_one(".title").text.strip()
    precio_raw = item.select_one(".price").text.strip()
    marca = item.select_one(".brand").text.strip()

    # Limpieza de cadena a flotante
    precio_clean = float(precio_raw.replace("$", "").replace(",", "").strip())

    datos_preparados.append({
        "producto": nombre,
        "precio_competidor": precio_clean,
        "marca": marca
    })

# 3. Conversión a DataFrame
df_competencia = pd.DataFrame(datos_preparados)
print("=== Datos Limpios Extraídos del Competidor ===")
print(df_competencia)

# 4. Archivo e inserción en Base de Datos SQLite (competidores_db.sqlite)
conexion = sqlite3.connect("competidores_db.sqlite")

# Escribir la tabla relacional 'precios_competencia'
df_competencia.to_sql("precios_competencia", conexion, if_exists="replace", index=False)

# 5. Consulta de Inteligencia Competitiva con SQL desde Python
query_analisis = """
SELECT producto, precio_competidor, marca
FROM precios_competencia
WHERE precio_competidor < 50000.0
ORDER BY precio_competidor ASC;
"""

df_oportunidades = pd.read_sql(query_analisis, conexion)
conexion.close()

print("\n=== Oportunidades Competitivas (Productos < $50,000) ===")
print(df_oportunidades)

=== Datos Limpios Extraídos del Competidor ===
                          producto  precio_competidor        marca
0       Silla Ergonómica Gamer Pro            45999.0  Marca Alpha
1  Escritorio Regulable Motorizado           120500.5  Marca Alpha
2             Soporte Monitor Dual            18200.0   Marca Beta

=== Oportunidades Competitivas (Productos < $50,000) ===
                     producto  precio_competidor        marca
0        Soporte Monitor Dual            18200.0   Marca Beta
1  Silla Ergonómica Gamer Pro            45999.0  Marca Alpha


---

## Desafío Práctico (Trabajo Práctico 5)

**Consigna:**
1. Generar un archivo HTML local llamado `competidores.html` (usando `%%writefile`) que simule la página de catálogo de un competidor con la siguiente estructura:
   - Un título principal de la tienda.
   - Al menos 4 tarjetas de productos, donde cada una tenga:
     - Nombre del producto.
     - Precio actual (ej. `$25.00`).
     - Precio anterior tachado (opcional, ej. `$30.00`).
     - Categoría del producto (ej. `"Tecnología"`, `"Hogar"`).
2. Escribir un script con **BeautifulSoup** que lea este archivo local y extraiga la información de todos los productos.
3. Guardar la información extraída en un **DataFrame de Pandas** y limpiar los campos (ej. remover el símbolo de moneda de los precios y convertirlos a float).
4. Guardar los resultados en una tabla llamada `precios_competencia` dentro de una base de datos local SQLite llamada `competidores_int.db`.
5. Ejecutar una consulta SQL simple desde Python usando Pandas (`pd.read_sql`) para mostrar los productos que tienen un precio menor a $15.000 (o 15.00 según los precios simulados).


In [ ]:
# --- Escribí tu resolución acá ---

# 1. Crear competidores.html
# %%writefile competidores.html
# ...

# 2. Leer y parsear con BeautifulSoup
# ...

# 3. Estructurar en Pandas DataFrame y limpiar datos
# ...

# 4. Guardar en SQLite
# ...

# 5. Consulta y verificación
# ...


In [14]:
%%writefile competidores.html
<!DOCTYPE html>
<html lang="es">
<head>
    <meta charset="UTF-8">
    <title>Tienda Competidor - Catálogo de Ofertas</title>
</head>
<body>
    <h1 class="titulo-tienda">Catálogo Oficial de Productos</h1>

    <div class="catalogo">
        <div class="card-producto">
            <h2 class="nombre-producto">Mouse Inalámbrico Pro</h2>
            <span class="precio-actual">$12.50</span>
            <span class="precio-anterior">$18.00</span>
            <span class="categoria">Tecnología</span>
        </div>

        <div class="card-producto">
            <h2 class="nombre-producto">Teclado Mecánico RGB</h2>
            <span class="precio-actual">$45.00</span>
            <span class="precio-anterior">$60.00</span>
            <span class="categoria">Tecnología</span>
        </div>

        <div class="card-producto">
            <h2 class="nombre-producto">Lámpara LED de Escritorio</h2>
            <span class="precio-actual">$14.99</span>
            <span class="categoria">Hogar</span>
        </div>

        <div class="card-producto">
            <h2 class="nombre-producto">Soporte Ergonómico para Notebook</h2>
            <span class="precio-actual">$8.50</span>
            <span class="precio-anterior">$12.00</span>
            <span class="categoria">Tecnología</span>
        </div>

        <div class="card-producto">
            <h2 class="nombre-producto">Silla de Oficina Ejecutiva</h2>
            <span class="precio-actual">$85.00</span>
            <span class="categoria">Hogar</span>
        </div>
    </div>
</body>
</html>

Writing competidores.html


##Celda 2: Extracción con BeautifulSoup, Limpieza, Persistencia en SQLite y Consulta SQL

In [15]:
import sqlite3
import pandas as pd
from bs4 import BeautifulSoup

# ==========================================
# 1. Leer y parsear el archivo HTML local
# ==========================================
with open("competidores.html", "r", encoding="utf-8") as f:
    soup = BeautifulSoup(f, "html.parser")

productos_extraidos = []

# Iterar sobre cada tarjeta de producto
for card in soup.select("div.card-producto"):
    # Extracción de campos
    nombre = card.select_one(".nombre-producto").text.strip()
    precio_actual_raw = card.select_one(".precio-actual").text.strip()

    # Campo opcional: precio anterior
    precio_ant_el = card.select_one(".precio-anterior")
    precio_anterior_raw = precio_ant_el.text.strip() if precio_ant_el else None

    categoria = card.select_one(".categoria").text.strip()

    # ==========================================
    # 2. Limpieza y transformación de datos
    # ==========================================
    # Remover el símbolo '$' y convertir a float
    precio_actual = float(precio_actual_raw.replace("$", "").replace(",", "").strip())

    precio_anterior = (
        float(precio_anterior_raw.replace("$", "").replace(",", "").strip())
        if precio_anterior_raw
        else None
    )

    productos_extraidos.append({
        "producto": nombre,
        "precio_actual": precio_actual,
        "precio_anterior": precio_anterior,
        "categoria": categoria
    })

# Guardar en DataFrame de Pandas
df_productos = pd.DataFrame(productos_extraidos)

print("=== 1. DataFrame Limpio ===")
print(df_productos)
print("\n" + "="*50 + "\n")

# ==========================================
# 3. Guardar en Base de Datos SQLite
# ==========================================
nombre_db = "competidores_int.db"
tabla_db = "precios_competencia"

conexion = sqlite3.connect(nombre_db)

# Guardar el DataFrame como tabla en SQLite
df_productos.to_sql(tabla_db, conexion, if_exists="replace", index=False)
conexion.close()

print(f"Éxito: Datos guardados en la tabla '{tabla_db}' dentro de '{nombre_db}'.")
print("\n" + "="*50 + "\n")

# ==========================================
# 4. Consulta SQL usando Pandas (pd.read_sql)
# ==========================================
conexion = sqlite3.connect(nombre_db)

# Consulta SQL: Buscar productos con precio menor a 15.00
query = f"""
SELECT producto, precio_actual, categoria
FROM {tabla_db}
WHERE precio_actual < 15.00
ORDER BY precio_actual ASC;
"""

df_filtrado = pd.read_sql(query, conexion)
conexion.close()

print("=== 2. Productos con Precio Menor a $15.00 (Resultado SQL) ===")
print(df_filtrado)

=== 1. DataFrame Limpio ===
                           producto  precio_actual  precio_anterior  \
0             Mouse Inalámbrico Pro          12.50             18.0   
1              Teclado Mecánico RGB          45.00             60.0   
2         Lámpara LED de Escritorio          14.99              NaN   
3  Soporte Ergonómico para Notebook           8.50             12.0   
4        Silla de Oficina Ejecutiva          85.00              NaN   

    categoria  
0  Tecnología  
1  Tecnología  
2       Hogar  
3  Tecnología  
4       Hogar  


Éxito: Datos guardados en la tabla 'precios_competencia' dentro de 'competidores_int.db'.


=== 2. Productos con Precio Menor a $15.00 (Resultado SQL) ===
                           producto  precio_actual   categoria
0  Soporte Ergonómico para Notebook           8.50  Tecnología
1             Mouse Inalámbrico Pro          12.50  Tecnología
2         Lámpara LED de Escritorio          14.99       Hogar
